# Train the Ultralytics YOLOv8n baselines

One notebook for both reference models of the comparison (see the README's Goal):

- `PRETRAINED = False` -> **YOLOv8n from scratch** (`yolov8n.yaml`, random init)
- `PRETRAINED = True`  -> **YOLOv8n fine-tuned** from COCO-pretrained weights (`yolov8n.pt`)

What's matched to this repo's own model, and what isn't -- on purpose:

| matched (defines the task) | each model's own (defines the method) |
|---|---|
| the exact same data split (the saved `manifest.json`) | training recipe: Ultralytics' defaults (augmentation incl. mosaic, optimizer, LR schedule, loss) |
| input size 320 | post-processing (NMS etc.) |
| 100-epoch budget | |
| best checkpoint chosen on val | |

Ultralytics prints its own mAP during training. **Those numbers aren't comparable** with
this repo's (different matching/AP code) -- the final comparison scores all three models
with the same code, on the test split, in a separate notebook.

## 1. Settings + mount Drive

In [ ]:
PRETRAINED = False   # False: from scratch, True: fine-tune COCO weights

from google.colab import drive
drive.mount("/content/drive")

RUN_ROOT = "/content/drive/MyDrive/yolo-edge-quantization"
# one Drive folder per model: rerunning with the same name RESUMES that training
RUN_NAME = "yolov8n_pretrained" if PRETRAINED else "yolov8n_scratch"
# the split all three models share -- saved by train_own_model.ipynb with run04
MANIFEST_RUN = "run04_long"

runDir = f"{RUN_ROOT}/{RUN_NAME}"
print("training:", RUN_NAME, "->", runDir)

## 2. Download the dataset

The manifest stores absolute image paths, so the images must be at the same location as
when it was built -- same Roboflow download, same `/content` working directory.

In [ ]:
!pip install -q roboflow

from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("efe-tatar").project("yolo-to-coco-traffic-signs-pmo2q")
dataset = project.version(1).download("coco")
print(dataset.location)

## 3. Clone (or update) the repo, install Ultralytics

Ultralytics is pinned so a resumed or repeated run uses the same code.

In [ ]:
import os
import sys

repoDirectory = "/content/yolo-edge-quantization"
if not os.path.isdir(repoDirectory):
    !git clone -q https://github.com/efe-tatar/yolo-edge-quantization.git {repoDirectory}
else:
    !git -C {repoDirectory} pull -q

!pip install -q -r {repoDirectory}/requirements.txt
!pip install -q ultralytics==8.3.40
!git -C {repoDirectory} log --oneline -1

sys.path.insert(0, repoDirectory)

## 4. Load the shared split

The same `manifest.json` and `classes.json` this repo's own model trained on -- copied
into this run's folder too, so each model's Drive folder records the split it used.

In [ ]:
import json
import shutil

manifestPath = f"{RUN_ROOT}/{MANIFEST_RUN}/manifest.json"
classesPath = f"{RUN_ROOT}/{MANIFEST_RUN}/classes.json"
with open(manifestPath) as f:
    manifest = json.load(f)
with open(classesPath) as f:
    classNames = json.load(f)

for setName in ["train", "valid", "test"]:
    print(setName, ":", len(manifest[setName]), "images")
print(len(classNames), "classes")

missing = [s["image_path"] for split in manifest.values() for s in split if not os.path.exists(s["image_path"])]
assert not missing, f"{len(missing)} manifest images not found, e.g. {missing[0]}"

os.makedirs(runDir, exist_ok=True)
shutil.copyfile(manifestPath, f"{runDir}/manifest.json")
shutil.copyfile(classesPath, f"{runDir}/classes.json")

## 5. Convert to Ultralytics' format

`data/yolo_format.py` (tested): copies the images to the VM's local disk and writes one
label file per image (`class cx cy w h`, normalized) plus `data.yaml`. Rebuilt every
session -- it's fast, and a resumed training expects it at this same path.

In [ ]:
from data.yolo_format import write_yolo_dataset

yoloDatasetDir = "/content/yolo_dataset"
dataYaml = write_yolo_dataset(manifest, classNames, yoloDatasetDir)

for split in ["train", "val", "test"]:
    print(split, ":", len(os.listdir(f"{yoloDatasetDir}/images/{split}")), "images")

## 6. Train

Ultralytics' default recipe; only the settings that define the task are set here.

- `epochs=100` is a **cap**: `best.pt` is the best epoch on val (Ultralytics validates
  every epoch), so later overfitting can't hurt the result, only cost time.
- `patience`: stop after that many epochs without val improvement -- 20 when fine-tuning
  (pretrained models usually peak early), effectively off from scratch, like this repo's
  own 100-epoch run.
- Checkpoints go straight to Drive (`runDir/train/weights`); rerunning this cell after a
  dropped session resumes from `last.pt`.

**`DISCONNECT_WHEN_DONE`**: when this cell ends -- finished *or* crashed -- flush Drive
and release the runtime, so an idle GPU doesn't keep burning compute units.

In [ ]:
import torch
from ultralytics import YOLO

epochs = 100
patience = 20 if PRETRAINED else epochs
batchSize = 16
imgSize = 320
DISCONNECT_WHEN_DONE = True

lastPt = f"{runDir}/train/weights/last.pt"

try:
    if os.path.exists(lastPt):
        # Ultralytics marks a finished run's checkpoint with epoch = -1
        finished = torch.load(lastPt, map_location="cpu", weights_only=False).get("epoch", -1) == -1
        if finished:
            print("training already finished:", lastPt)
        else:
            print("resuming from", lastPt)
            YOLO(lastPt).train(resume=True)
    else:
        model = YOLO("yolov8n.pt" if PRETRAINED else "yolov8n.yaml")
        model.train(
            data=dataYaml, imgsz=imgSize, epochs=epochs, patience=patience, batch=batchSize,
            pretrained=PRETRAINED, project=runDir, name="train", exist_ok=True,
            seed=42, workers=2,
        )
finally:
    if DISCONNECT_WHEN_DONE:
        from google.colab import runtime
        drive.flush_and_unmount()  # make sure the last checkpoint is written to Drive first
        runtime.unassign()

## 7. Training summary

Ultralytics' own per-epoch log (`results.csv`) and curves (`results.png`). **Not the
comparison numbers** -- those come from the comparison notebook. After a
`DISCONNECT_WHEN_DONE` run, rerun section 1 first (a CPU runtime is enough).

In [ ]:
import pandas as pd
from IPython.display import Image, display

results = pd.read_csv(f"{runDir}/train/results.csv")
results.columns = [c.strip() for c in results.columns]
cols = ["epoch", "metrics/mAP50(B)", "metrics/mAP50-95(B)", "val/box_loss", "val/cls_loss"]

best = results.loc[results["metrics/mAP50-95(B)"].idxmax()]
print(f"{len(results)} epochs trained; best mAP50-95 (Ultralytics' own) at epoch {int(best['epoch'])}")
display(results[cols].tail(10))
display(Image(f"{runDir}/train/results.png"))